In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import hvplot.pandas
import holoviews as hv
from workspace_dani.data_preprocessing.utils import data_handling

hv.extension("bokeh")

In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("build") / Path(
    "2024-03-26") / Path("build_dataset_cats.parquet")
dataset_path

In [ ]:
df_patients = data_handling.load_parquet_table(parquet_path=dataset_path, debug_mode=False, backend="numpy_nullable")


### Setting the age "> 89" to 90 since we can't evaluate it properly otherwise, it's 90 since we can assume that they are at least 90 and we will just group them together

In [ ]:
df_patients["age"] = df_patients["age"].replace("> 89", 90)

In [ ]:
patient_ids_dir: Path = project_root_path / Path("intermediary_data") / Path("patient_unit_stay_ids") / Path("cohorts")


In [ ]:
path_valid_apache_ids: Path = patient_ids_dir / Path("valid_apache") / Path("valid_ids_from_apache.parquet")

path_discharge_offset_ids: Path = patient_ids_dir / Path("unit_discharge_offset") / Path(
    "included_ids_unit_discharge_offset.parquet")

path_visit_number_ids: Path = patient_ids_dir / Path("visit_number") / Path("included_ids_visit_number.parquet")

path_admit_source_ids: Path = patient_ids_dir / Path("admit_source") / Path("included_ids_admit_source.parquet")

path_icu_type_a_ids: Path = patient_ids_dir / Path("icu_types") / Path("included_ids_unit_type_a.parquet")

path_icu_type_b_ids: Path = patient_ids_dir / Path("icu_types") / Path("included_ids_unit_type_b.parquet")

sepsis_to_include_path = project_root_path / Path("intermediary_data") / Path("patient_unit_stay_ids") / Path(
    "cohorts") / Path("sepsis") / Path(
    "included_ids_sepsis.parquet")
sepsis_to_exclude_path = project_root_path / Path("intermediary_data") / Path("patient_unit_stay_ids") / Path(
    "cohorts") / Path("sepsis") / Path(
    "excluded_ids_sepsis.parquet")

labels_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "sepsis_labels_concise.parquet")




In [ ]:
df_valid_apache = data_handling.load_parquet_table(parquet_path=path_valid_apache_ids, debug_mode=False,
                                                   backend="numpy_nullable", verbose=False)
df_discharge_offset = data_handling.load_parquet_table(parquet_path=path_discharge_offset_ids, debug_mode=False,
                                                       backend="numpy_nullable", verbose=False)
df_visit_number = data_handling.load_parquet_table(parquet_path=path_visit_number_ids, debug_mode=False,
                                                   backend="numpy_nullable", verbose=False)
df_admit_source = data_handling.load_parquet_table(parquet_path=path_admit_source_ids, debug_mode=False,
                                                   backend="numpy_nullable", verbose=False)
df_icu_type_a = data_handling.load_parquet_table(parquet_path=path_icu_type_a_ids, debug_mode=False,
                                                 backend="numpy_nullable", verbose=False)
df_icu_type_b = data_handling.load_parquet_table(parquet_path=path_icu_type_b_ids, debug_mode=False,
                                                 backend="numpy_nullable", verbose=False)

df_sepsis_ids_to_include = data_handling.load_parquet_table(parquet_path=sepsis_to_include_path, debug_mode=False,
                                                            backend="numpy_nullable", verbose=False)
df_sepsis_ids_to_exclude = data_handling.load_parquet_table(parquet_path=sepsis_to_exclude_path, debug_mode=False,
                                                            backend="numpy_nullable", verbose=False)

df_sepsis = data_handling.load_parquet_table(parquet_path=labels_path, debug_mode=False, backend="numpy_nullable",
                                             verbose=False)


###### Note that the df_sepsis still contains all sepsis labels that have been generated, labels don't fulfil our requirements are discarded via the exluded sepsis patient unit stay ids

In [ ]:
ids_type_a_dfs: list = [df_valid_apache, df_discharge_offset, df_visit_number, df_admit_source, df_icu_type_a]
ids_type_b_dfs: list = [df_valid_apache, df_discharge_offset, df_visit_number, df_admit_source, df_icu_type_b]


In [ ]:
individual_cohorts_type_a_ids: list = [frame.iloc[:, 0].to_list() for frame in ids_type_a_dfs]
individual_cohorts_type_b_ids: list = [frame.iloc[:, 0].to_list() for frame in ids_type_b_dfs]

### Determining the intersection between each set of patient unit stay ids

In [ ]:
from functools import reduce

ids_cohort_a_all = sorted(reduce(lambda x, y: set(x) & set(y), individual_cohorts_type_a_ids))
ids_cohort_b_all = sorted(reduce(lambda x, y: set(x) & set(y), individual_cohorts_type_b_ids))

#### Excluding patients with a sepsis label that fulfill our cohort requirements but the sepsis label doesn't fall within the 168 h timeframe!


In [ ]:

sepsis_ids_to_exclude: list = df_sepsis_ids_to_exclude.iloc[:, 0].to_list()

ids_cohort_a = sorted(set(ids_cohort_a_all) - set(sepsis_ids_to_exclude))
ids_cohort_b = sorted(set(ids_cohort_b_all) - set(sepsis_ids_to_exclude))

In [ ]:
df_patients_cohort_a = df_patients[df_patients["patientunitstayid"].isin(ids_cohort_a)]
df_patients_cohort_b = df_patients[df_patients["patientunitstayid"].isin(ids_cohort_b)]

In [ ]:
cohort_a_reduction_due_to_sepsis_req = len(ids_cohort_a_all) - len(ids_cohort_a)
cohort_b_reduction_due_to_sepsis_req = len(ids_cohort_b_all) - len(ids_cohort_b)

cohort_a_reduction_due_to_sepsis_req

In [ ]:
cohort_b_reduction_due_to_sepsis_req

##### Only an minor reduction of cohort size!

In [ ]:
df_patients.head()

In [ ]:
cohort_characterization_variables = ["apachedxgroup", "physicianspeciality", "acutephysiologyscore", "apachescore",
                                     "predictedicumortality", "actualicumortality", "predictediculos", "actualiculos",
                                     "predictedhospitalmortality", "actualhospitalmortality", "bedcount",
                                     "electivesurgery", "gender", "age", "admissionweight", "admissionheight",
                                     "ethnicity", "hospitalid", "unittype",
                                     "unitvisitnumber", "unitadmitsource", "unitdischargeoffset",
                                     "unitdischargelocation"]

In [ ]:
df_icu_type_a_char_variables = df_patients_cohort_a.loc[:, cohort_characterization_variables]
df_icu_type_b_char_variables = df_patients_cohort_b.loc[:, cohort_characterization_variables]


In [ ]:
df_icu_type_a_char_variables["age"] = df_icu_type_a_char_variables["age"].astype("int")
df_icu_type_a_char_variables["admissionweight"] = df_icu_type_a_char_variables["admissionweight"].astype("Float32")


In [ ]:
df_icu_type_a_char_variables.info()

In [ ]:
df_icu_type_a_char_variables.describe(include="all", percentiles=[0.01, 0.1, 0.9, 0.99])

### Cohort Size

In [ ]:
cohort_size = len(df_icu_type_a_char_variables)

### Frequencies for categorical data: apache admission dx, clinician specialty, mortality frequencies, elective surgery, gender, ethnicity, hospital id, icu unit type, admit source, unit dicharge location

In [ ]:
considered_categorical_data = [
    # "apachedxgroup", "physicianspeciality",
    "actualicumortality", "actualhospitalmortality", "electivesurgery", "gender",
    "ethnicity", "unittype", "unitadmitsource",
    "unitdischargelocation"
]

In [ ]:
list_freqs_cat = []
for var in considered_categorical_data:
    var_freq: pd.DataFrame = df_icu_type_a_char_variables.loc[:, var].value_counts().reset_index()
    var_freq.columns = [var, var + "_count"]

    var_freq_relative: pd.DataFrame = \
        df_icu_type_a_char_variables.loc[:, var].value_counts(normalize=True).reset_index()["proportion"]

    var_freq_relative = var_freq_relative.rename(var + "_proportion")

    var_freqs_absolute_relative = pd.concat([var_freq, var_freq_relative], axis=1)

    list_freqs_cat.append(var_freqs_absolute_relative)

In [ ]:
df_cohort_data_counts: pd.DataFrame = pd.concat(list_freqs_cat, axis=1, ignore_index=False)

In [ ]:
df_cohort_data_counts

In [ ]:
#df_icu_type_a_cat_data_counts.to_latex(index=True, na_rep="", float_format="{:.2f}".format)

### Median and IQR for numerical data cohort data


In [ ]:
df_iqr_numerical_cohort = df_icu_type_a_char_variables.quantile(q=[0.001, 0.25, 0.5, 0.75, 0.999], numeric_only=True)

In [ ]:
df_iqr_numerical_cohort["unitdischargeoffset"] = pd.to_timedelta(
    df_iqr_numerical_cohort["unitdischargeoffset"], unit="minutes")

### There's basically no information on elective surgery, so we will drop that column


In [ ]:
df_iqr_numerical_cohort = df_iqr_numerical_cohort.drop(columns=["electivesurgery"])

In [ ]:
df_iqr_numerical_cohort

#### Taking a look at sepsis characteristics!


In [ ]:
sepsis_ids_to_include = df_sepsis_ids_to_include.iloc[:, 0].to_list()
df_patients_cohort_a["sepsis"] = df_patients_cohort_a.loc[:, "patientunitstayid"].isin(sepsis_ids_to_include)
df_patients_cohort_b["sepsis"] = df_patients_cohort_b.loc[:, "patientunitstayid"].isin(sepsis_ids_to_include)


In [ ]:
df_cohort_a_sepsis = df_patients_cohort_a["sepsis"]
df_cohort_b_sepsis = df_patients_cohort_b["sepsis"]


In [ ]:
df_cohort_a_sepsis.describe()

In [ ]:
df_cohort_b_sepsis.describe()

In [ ]:
df_cohort_a_sepsis.value_counts()

In [ ]:
df_cohort_b_sepsis.value_counts()

In [ ]:
df_cohort_a_sepsis.value_counts(normalize=True)

In [ ]:
df_cohort_b_sepsis.value_counts(normalize=True)


In [ ]:
df_cohort_a_sepsis_characteristics = df_sepsis[
    df_sepsis["patientunitstayid"].isin(df_patients_cohort_a["patientunitstayid"])]

In [ ]:
df_cohort_a_sepsis_characteristics[["sepsis_offset", "time_delta"]].describe()

### Characterization of the assoicated SOFA score of the patients

In [ ]:
sofa_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "sofa_data_15min.parquet")

In [ ]:
df_data_sofa = data_handling.load_parquet_table(parquet_path=sofa_path, debug_mode=False, backend="numpy_nullable")

In [ ]:
df_data_sofa_cohort_a: pd.DataFrame = df_data_sofa[df_data_sofa["patientunitstayid"].isin(df_patients_cohort_a["patientunitstayid"])]

In [ ]:
df_data_sofa_cohort_a.describe()